# 31 · LangGraph 状态机 + 检查点 + 中断恢复

> **学习目标**：把「ReAct loop（25 号）」升级到「**显式状态机**（LangGraph）」。理解 nodes / edges / conditional edges 三件套，跑通 checkpoint（断点续跑）+ interrupt（人工介入）两大杀手特性。
>
> **预备**：25、30 跑过。本机 `langgraph` 1.1+ 已装。
>
> **为什么重要**：当 Agent 流程超过 3 步、有条件分支、需要中途暂停（HITL）时，**状态机比 while-loop 清晰 10 倍**。LangGraph 是 LangChain 团队推荐的 production-grade Agent 编排框架。

In [ ]:
import langgraph, importlib.metadata
print('langgraph:', importlib.metadata.version('langgraph'))
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

## 1. 状态机 vs ReAct loop —— 何时该升级

| | ReAct while-loop（25 号） | LangGraph 状态机 |
|---|--------------------------|----------------|
| 流程清晰度 | 流程藏在代码里 | **图可视化** |
| 条件分支 | if/else 散在循环里 | 显式 conditional edge |
| checkpoint | 自己实现 | **内置** `MemorySaver` / Postgres / Redis |
| 人工介入 (HITL) | 自己处理 | 内置 `interrupt()` |
| 学习曲线 | 0 | 中（要学新 API）|
| 调试 | print + trace | LangGraph Studio 可视化 |

**经验**：3 步以下 → ReAct 够；3 步以上 + 分支 + 中断 → 状态机。

## 2. 最小状态机 —— 3 节点 + 1 条件边

**场景**：写文章 Agent。流程 = **planner → writer → critic**，critic 评审若不通过 → 回 writer 修，最多 3 轮。

```
        START
          │
          ▼
       planner
          │
          ▼
        writer ◄──┐
          │       │
          ▼       │
       critic ────┘ (need_revision=True 且 round < 3)
          │
          ▼
         END
```

In [ ]:
# 1. 定义 state schema（TypedDict —— 类型化字典）
class WriteState(TypedDict):
    topic: str
    outline: str
    draft: str
    feedback: str
    round: int
    final: str

# 2. 定义节点 —— 每个节点是「读 state → 改 state」的函数
def node_planner(state: WriteState) -> dict:
    print(f'  [planner] topic={state["topic"]!r}')
    outline = f'1. 引言：{state["topic"]}定义\n2. 主体：{state["topic"]}核心机制\n3. 结尾：{state["topic"]}的应用'
    return {'outline': outline, 'round': 0}

def node_writer(state: WriteState) -> dict:
    round_num = state.get('round', 0) + 1
    base = f'根据大纲 {state["outline"][:30]}... 写出一段关于 {state["topic"]} 的内容。'
    if state.get('feedback'):
        base = f'[根据上轮反馈 {state["feedback"][:30]!r} 修订] ' + base
    print(f'  [writer  ] round {round_num} 完成 (草稿长度 {len(base)} 字)')
    return {'draft': base + '（约 100 字稿，round=' + str(round_num) + '）', 'round': round_num}

def node_critic(state: WriteState) -> dict:
    draft = state['draft']
    issues = []
    if state['round'] < 2:   # 头两轮强制提改进意见
        issues.append('缺少具体例子')
    if len(draft) < 60:
        issues.append('过短')
    if issues:
        fb = '；'.join(issues)
        print(f'  [critic  ] 不通过: {fb}')
        return {'feedback': fb}
    print(f'  [critic  ] 通过')
    return {'feedback': '', 'final': state['draft']}

# 3. 条件边 —— critic 后决定回 writer 还是结束
def route_after_critic(state: WriteState) -> Literal['writer', 'end']:
    if state.get('feedback') and state['round'] < 3:
        return 'writer'
    return 'end'

In [ ]:
# 4. 装配状态图
builder = StateGraph(WriteState)
builder.add_node('planner', node_planner)
builder.add_node('writer',  node_writer)
builder.add_node('critic',  node_critic)

builder.add_edge(START, 'planner')
builder.add_edge('planner', 'writer')
builder.add_edge('writer', 'critic')
builder.add_conditional_edges('critic', route_after_critic, {'writer': 'writer', 'end': END})

graph = builder.compile()
print('状态图已编译')

# Mermaid 可视化（如果你装了 graphviz 可输出 PNG）
try:
    print('\nMermaid:')
    print(graph.get_graph().draw_mermaid())
except Exception as e:
    print(f'(skip mermaid: {e})')

In [ ]:
# 5. 跑
print('=' * 50)
result = graph.invoke({'topic': 'RAG', 'outline': '', 'draft': '', 'feedback': '', 'round': 0, 'final': ''})
print('=' * 50)
print(f'\n最终结果（{result["round"]} 轮）:')
print(f'  draft: {result["draft"]}')
print(f'  final: {result["final"]!r}')

## 3. Checkpoint —— 断点续跑

**场景**：长任务跑到一半挂了。希望重启后**接着跑**而不是从头来。

**LangGraph 做法**：编译时加 `checkpointer`，调用时给 `thread_id`，每步 state 自动写 checkpoint。

In [ ]:
checkpointer = MemorySaver()   # 演示用内存版；生产用 PostgresSaver / RedisSaver
graph_ck = builder.compile(checkpointer=checkpointer)

config = {'configurable': {'thread_id': 'session-001'}}

# 第一次跑（完整跑完）
result = graph_ck.invoke({'topic': 'Agent', 'outline': '', 'draft': '', 'feedback': '', 'round': 0, 'final': ''}, config=config)
print(f'\n第一次跑完，最终 round={result["round"]}')

# 检查 checkpointer 里有什么
history = list(graph_ck.get_state_history(config))
print(f'\n这条 thread 一共写了 {len(history)} 个 checkpoint')
for i, ckpt in enumerate(history[:3]):
    print(f'  checkpoint {i}: next={ckpt.next}  round={ckpt.values.get("round")}')

# 同 thread 继续 invoke：会从上次 final state 接着跑（无新输入则空闲）
print('\n（同 thread_id 再 invoke 表示「接着这次会话」，新 thread_id 才是新会话）')

In [ ]:
# 演示「跑一步停一步」用 stream 模式 + checkpoint 自动持久化
from copy import deepcopy

fresh_cfg = {'configurable': {'thread_id': 'session-002'}}
initial = {'topic': 'LangGraph', 'outline': '', 'draft': '', 'feedback': '', 'round': 0, 'final': ''}

print('--- 模拟「跑两步就停」---')
stream_iter = graph_ck.stream(initial, config=fresh_cfg, stream_mode='updates')
step = 0
for event in stream_iter:
    step += 1
    print(f'  step {step}: 节点 {list(event.keys())} 已执行')
    if step >= 2:
        print('  ⏸ 主动停下（模拟服务挂了 / 用户关了浏览器）')
        break

# 检查 state
state_now = graph_ck.get_state(fresh_cfg)
print(f'\n挂的时候 state: round={state_now.values.get("round")}, next 节点={state_now.next}')

# 重启 / 续跑：用同 thread_id 调 invoke（input=None 表示「从 checkpoint 续」）
print('\n--- 续跑：同 thread_id ---')
result = graph_ck.invoke(None, config=fresh_cfg)
print(f'最终结果: round={result["round"]}, final={result["final"][:50]!r}')

## 4. Interrupt —— 人工介入 (HITL, Human-in-the-Loop)

**场景**：critic 觉得稿子有争议，**让人工评审**再决定要不要继续。

**LangGraph 做法**：在节点里调 `interrupt(payload)` —— 图会暂停、把 payload 抛出，等外部 `Command(resume=...)` 续跑。

In [ ]:
def node_critic_hitl(state: WriteState) -> dict:
    """critic 变形：当 round >= 2 时请求人工评审。"""
    draft = state['draft']
    if state['round'] >= 2:
        # 暂停，把要审的内容抛给外部
        human_decision = interrupt({
            'question': '这版稿子可以发布吗？请回 approve / revise / abort',
            'draft_preview': draft[:80],
        })
        print(f'  [critic ] 人工决定: {human_decision}')
        if human_decision == 'approve':
            return {'feedback': '', 'final': draft}
        if human_decision == 'abort':
            return {'feedback': '', 'final': '(用户中止)'}
        return {'feedback': '人工要求继续修订'}
    # 自动评审（同前）
    fb = '缺例子' if state['round'] < 2 else ''
    return {'feedback': fb}

builder2 = StateGraph(WriteState)
builder2.add_node('planner', node_planner)
builder2.add_node('writer', node_writer)
builder2.add_node('critic', node_critic_hitl)
builder2.add_edge(START, 'planner')
builder2.add_edge('planner', 'writer')
builder2.add_edge('writer', 'critic')
builder2.add_conditional_edges('critic', route_after_critic, {'writer': 'writer', 'end': END})
graph_hitl = builder2.compile(checkpointer=MemorySaver())

hitl_cfg = {'configurable': {'thread_id': 'hitl-001'}}
initial = {'topic': 'HITL', 'outline': '', 'draft': '', 'feedback': '', 'round': 0, 'final': ''}

print('--- 跑到 critic 时会触发 interrupt ---')
result = graph_hitl.invoke(initial, config=hitl_cfg)
# 因为 interrupt 抛出，invoke 提前返回
if '__interrupt__' in result:
    info = result['__interrupt__'][0].value
    print(f'\n⏸ 图暂停，等待人工。Payload:')
    print(f'  question: {info["question"]}')
    print(f'  draft:    {info["draft_preview"]!r}')
else:
    print(f'\n（未触发 interrupt，可能 round 没到 2。final={result.get("final")!r}）')

In [ ]:
# 模拟人工回 'approve' 续跑
print('--- 人工回 approve ---')
result = graph_hitl.invoke(Command(resume='approve'), config=hitl_cfg)
print(f'\n最终 final: {result.get("final")!r}')
print(f'round: {result.get("round")}')

## 5. 把 ReAct loop 也建成 LangGraph —— 二者结合

**生产实战**：状态机的 node 里**装一个 ReAct loop**（25 号风格 Agent）。state 管「高阶流程」，ReAct 管「具体怎么用工具」。

示意结构：
```
       START
         │
         ▼
    [分类节点]  →  自己决定走哪条分支
         │
    ┌────┼────┐
    ▼    ▼    ▼
  [RAG节点] [Agent节点] [拒答节点]
    ↑
    └── 内部跑 25 号风格 ReAct loop
```

**优点**：高阶流程一目了然，内部细节归节点。

## 深入思考

1. **state 用 TypedDict 还是 Pydantic？**
   - 都行。TypedDict 轻量；Pydantic 强校验、能算字段是否变化。**生产推荐 Pydantic** —— state 错了能立刻 raise。
2. **checkpointer 选 MemorySaver / Postgres / Redis？**
   - 本机 demo MemorySaver；生产 Postgres（关系型 + 事务）/ Redis（高吞吐）。**有 checkpointer = 长会话能恢复**。
3. **interrupt 抛出后 invoke 会怎样？**
   - 返回的 result 里含 `__interrupt__` 字段。下次必须用 `Command(resume=...)` 续，**不能直接 invoke()**。
4. **状态图的「分支」能不能有 cycle？**
   - 能（我们的 writer ↔ critic 就是）。LangGraph 自动检测 + 用 max_step 防死循环（默认 25）。
5. **vs 手写 ReAct loop 复杂度怎么衡量？**
   - 3 节点以下：手写 loop 代码更短；4 节点 + 条件分支：LangGraph 反而更短。**关键看分支数 + 是否需要 checkpoint / HITL**。

**改一改**：
- 把 `route_after_critic` 的最大 round 改到 5，观察 cycle 跑几轮
- 加一个 `node_save` 节点，把 final 写到沙箱文件
- 把 HITL demo 的人工回复改成 `revise`，看会不会回到 writer

## 自检 ✅

- [ ] 默写 LangGraph 4 件套：StateGraph / add_node / add_edge / add_conditional_edges
- [ ] 解释 checkpoint + thread_id 的关系
- [ ] 解释 interrupt + Command(resume=...) 的搭配用法
- [ ] 给一个 3 节点 + 1 分支的需求，能 10 分钟内画出状态图
- [ ] 解释「state 机比 while-loop 何时更优」的 2 个关键判据

## 下一步

进入 Stage 4 → [`../stage4_专家/32_mini_agent_framework.ipynb`](../stage4_专家/32_mini_agent_framework.ipynb)